# Passos Mágicos — Modelo de Risco de Defasagem

**Desenho:** indicadores do ano **T** → defasagem no ano **T+1** (anti-vazamento).

## Correções em relação à versão anterior
- **Diagnóstico de leakage:** a base traz `Fase Ideal` e `Defasagem = Fase − Fase Ideal` em
  **100%** dos casos, com `Fase Ideal` determinada pela `Idade`. Prever a defasagem *do mesmo
  ano* a partir de Fase/Idade é reproduzir uma subtração — não é previsão.
- **IAN e INDE removidos** (derivados da Defasagem).
- **Alvo movido para T+1**, genuinamente desconhecido no momento da previsão.
- **Genero removido** — piorava o modelo e levanta questão de equidade.


In [ ]:
import re, warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, average_precision_score, recall_score,
                             precision_score, f1_score, confusion_matrix)
import joblib

warnings.filterwarnings('ignore')
DATA = Path('BASE DE DADOS PEDE 2024 - DATATHON.xlsx')
print('Arquivo encontrado:', DATA.exists())

## 1. Carregamento e normalização

In [ ]:
def fase_num(x):
    """Converte Fase para número. ALFA = 0."""
    if pd.isna(x): return np.nan
    s = str(x).upper().strip()
    if 'ALFA' in s: return 0.0
    m = re.search(r'(\d+)', s)
    return float(m.group(1)) if m else np.nan

def clean_pedra(v):
    if pd.isna(v): return np.nan
    s = str(v).strip()
    if s in ('', 'nan', 'NaN', 'None'): return np.nan
    s = s.title()
    return {'Ágata': 'Agata', 'Incluir': 'Outros'}.get(s, s)

xl = pd.ExcelFile(DATA)
frames = []
for sheet, year in [('PEDE2022', 2022), ('PEDE2023', 2023), ('PEDE2024', 2024)]:
    df = pd.read_excel(xl, sheet_name=sheet)
    df.columns = [str(c) for c in df.columns]
    df['Ano'] = year
    cm = {'Gênero': 'Genero'}
    if year == 2022:
        cm.update({'Defas': 'Defasagem', 'INDE 22': 'INDE', 'Pedra 22': 'Pedra', 'Idade 22': 'Idade'})
    elif year == 2023:
        cm.update({'INDE 2023': 'INDE', 'Pedra 2023': 'Pedra'})
    else:
        cm.update({'INDE 2024': 'INDE', 'Pedra 2024': 'Pedra'})
    df = df.rename(columns={k: v for k, v in cm.items() if k in df.columns})
    df['Fase_num'] = df['Fase'].apply(fase_num)
    if 'Pedra' in df.columns:
        df['Pedra'] = df['Pedra'].apply(clean_pedra)
    keep = ['Ano','RA','Fase_num','Genero','Idade','Ano ingresso','Pedra',
            'INDE','IAA','IEG','IPS','IPP','IDA','IPV','IAN','Defasagem']
    frames.append(df[[c for c in keep if c in df.columns]].copy())

base = pd.concat(frames, ignore_index=True)
for c in ['INDE','IAA','IEG','IPS','IPP','IDA','IPV','IAN','Defasagem','Idade','Ano ingresso','Fase_num']:
    base[c] = pd.to_numeric(base[c], errors='coerce')
base['tempo_prog'] = base['Ano'] - base['Ano ingresso']
print(base.shape)
base.head()

## 2. Diagnóstico de leakage

Duas checagens que motivam todo o desenho desta versão.

In [ ]:
# [1] Defasagem é calculada a partir de Fase e Fase Ideal?
for sheet, fideal, dcol in [('PEDE2022','Fase ideal','Defas'),
                            ('PEDE2023','Fase Ideal','Defasagem'),
                            ('PEDE2024','Fase Ideal','Defasagem')]:
    d = pd.read_excel(xl, sheet_name=sheet)
    f  = d['Fase'].apply(fase_num)
    fi = d[fideal].apply(fase_num)
    de = pd.to_numeric(d[dcol], errors='coerce')
    m = f.notna() & fi.notna() & de.notna()
    print(f'{sheet}: Defasagem == Fase - Fase_Ideal em {((f[m]-fi[m])==de[m]).mean()*100:.1f}% (n={m.sum()})')

In [ ]:
# [2] Prevendo a defasagem DO MESMO ANO: o modelo usa os indicadores ou só faz aritmética?
b = base.copy()
b['risco_mesmo_ano'] = (b['Defasagem'] < 0).astype(int)
tr_, te_ = b[b['Ano'].isin([2022, 2023])], b[b['Ano'] == 2024]

def experimento(num, cat, nome):
    steps = [('num', Pipeline([('i', SimpleImputer(strategy='median')),
                               ('s', StandardScaler())]), num)]
    if cat:
        steps.append(('cat', Pipeline([('i', SimpleImputer(strategy='most_frequent')),
                      ('o', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat))
    p = Pipeline([('pre', ColumnTransformer(steps)),
                  ('mdl', GradientBoostingClassifier(n_estimators=300, max_depth=4,
                           learning_rate=0.05, subsample=0.8, random_state=42))])
    p.fit(tr_[num + cat], tr_['risco_mesmo_ano'])
    pr = p.predict_proba(te_[num + cat])[:, 1]
    print(f'  {nome:44s} ROC-AUC={roc_auc_score(te_["risco_mesmo_ano"], pr):.4f}')

print('Prevendo a defasagem do MESMO ano (teste 2024):')
experimento(['Fase_num','Idade','IAA','IEG','IPS','IPP','IDA','IPV'], ['Pedra'],
            'A) Fase + Idade + indicadores')
experimento(['Fase_num','Idade'], [], 'B) SÓ Fase + Idade (zero indicadores)')
experimento(['IAA','IEG','IPS','IPP','IDA','IPV'], [], 'C) SÓ os indicadores')
print('\n>>> Se B ≈ A, o modelo está reproduzindo aritmética, não prevendo.')

## 3. Base de transições T → T+1

O alvo correto: o aluno estará defasado **no ano seguinte**?

In [ ]:
pares = []
for t in [2022, 2023]:
    a = base[base['Ano'] == t].copy()
    b_ = base[base['Ano'] == t + 1][['RA', 'Defasagem']].rename(columns={'Defasagem': 'Defasagem_T1'})
    j = a.merge(b_, on='RA', how='inner')
    j['transicao'] = f'{t}->{t+1}'
    pares.append(j)

tr = pd.concat(pares, ignore_index=True)
tr = tr[tr['Defasagem_T1'].notna()].copy()
tr['alvo'] = (tr['Defasagem_T1'] < 0).astype(int)
tr['risco_atual'] = (tr['Defasagem'] < 0).astype(int)

print(f'Transições: {len(tr)} | alunos únicos: {tr["RA"].nunique()}')
print(tr.groupby('transicao').agg(n=('RA','count'), taxa_alvo=('alvo','mean')).round(3))

## 4. Baselines honestos

Antes de comemorar qualquer AUC: **quanto o modelo agrega** sobre simplesmente supor que a
situação se mantém?

In [ ]:
train = tr[tr['transicao'] == '2022->2023']
test  = tr[tr['transicao'] == '2023->2024']

NUM = ['Defasagem','Fase_num','Idade','IAA','IEG','IPS','IPP','IDA','IPV','tempo_prog']
CAT = ['Pedra']
FEATURES = NUM + CAT

print(f'[0] Persistência pura: ROC-AUC={roc_auc_score(test["alvo"], test["risco_atual"]):.4f}')

def roda(num, cat, nome):
    steps = [('num', Pipeline([('i', SimpleImputer(strategy='median')),
                               ('s', StandardScaler())]), num)]
    if cat:
        steps.append(('cat', Pipeline([('i', SimpleImputer(strategy='most_frequent')),
                      ('o', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat))
    p = Pipeline([('pre', ColumnTransformer(steps)),
                  ('mdl', GradientBoostingClassifier(n_estimators=300, max_depth=3,
                           learning_rate=0.05, subsample=0.8, random_state=42))])
    p.fit(train[num + cat], train['alvo'])
    pr = p.predict_proba(test[num + cat])[:, 1]
    auc = roc_auc_score(test['alvo'], pr)
    print(f'  {nome:46s} ROC-AUC={auc:.4f}')
    return auc

a = roda(['Defasagem','Fase_num','Idade'], [], '[A] só estado atual')
b_ = roda(['IAA','IEG','IPS','IPP','IDA','IPV'], ['Pedra'], '[B] só indicadores')
c = roda(FEATURES, CAT, '[C] completo')
print(f'\n>>> Ganho dos indicadores sobre o estado atual: {c-a:+.4f} AUC')

## 5. Comparação de modelos e decisão sobre `Genero`

In [ ]:
def pipe(model, num, cat):
    steps = [('num', Pipeline([('i', SimpleImputer(strategy='median')),
                               ('s', StandardScaler())]), num)]
    if cat:
        steps.append(('cat', Pipeline([('i', SimpleImputer(strategy='most_frequent')),
                      ('o', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat))
    return Pipeline([('pre', ColumnTransformer(steps)), ('mdl', model)])

MODELS = {
    'LogisticRegression': LogisticRegression(max_iter=3000, class_weight='balanced', C=0.1),
    'RandomForest': RandomForestClassifier(n_estimators=400, max_depth=6, min_samples_leaf=15,
                                           class_weight='balanced', random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=300, max_depth=3,
                                           learning_rate=0.05, subsample=0.8, random_state=42),
}
for nome, mdl in MODELS.items():
    p = pipe(mdl, NUM, CAT); p.fit(train[FEATURES], train['alvo'])
    pr = p.predict_proba(test[FEATURES])[:, 1]
    print(f'  {nome:20s} ROC-AUC={roc_auc_score(test["alvo"], pr):.4f}  '
          f'PR-AUC={average_precision_score(test["alvo"], pr):.4f}')

print('\nImpacto de incluir Genero (decisão ética + evidência):')
for nome, cat in [('SEM Genero', ['Pedra']), ('COM Genero', ['Pedra', 'Genero'])]:
    p = pipe(MODELS['RandomForest'], NUM, cat); p.fit(train[NUM + cat], train['alvo'])
    pr = p.predict_proba(test[NUM + cat])[:, 1]
    print(f'  {nome:12s} PR-AUC={average_precision_score(test["alvo"], pr):.4f}')

## 6. Modelo final: RandomForest calibrado

In [ ]:
def build():
    pre = ColumnTransformer([
        ('num', Pipeline([('i', SimpleImputer(strategy='median')),
                          ('s', StandardScaler())]), NUM),
        ('cat', Pipeline([('i', SimpleImputer(strategy='most_frequent')),
                          ('o', OneHotEncoder(handle_unknown='ignore',
                                              sparse_output=False))]), CAT)])
    rf = RandomForestClassifier(n_estimators=400, max_depth=6, min_samples_leaf=15,
                                class_weight='balanced', random_state=42, n_jobs=-1)
    return Pipeline([('pre', pre), ('mdl', CalibratedClassifierCV(rf, method='sigmoid', cv=5))])

pipe_final = build()
pipe_final.fit(train[FEATURES], train['alvo'])
proba = pipe_final.predict_proba(test[FEATURES])[:, 1]

# limiar para recall >= 0.85
thr = 0.5
for t in np.arange(0.05, 0.95, 0.01):
    if recall_score(test['alvo'], (proba >= t).astype(int)) >= 0.85:
        thr = t
pred = (proba >= thr).astype(int)

metricas = {'AUC-PR': average_precision_score(test['alvo'], proba),
            'AUC-ROC': roc_auc_score(test['alvo'], proba),
            'Recall_risco': recall_score(test['alvo'], pred),
            'Precision_risco': precision_score(test['alvo'], pred),
            'F1': f1_score(test['alvo'], pred)}
print(f'Limiar (recall>=0.85): {thr:.2f}')
for k, v in metricas.items(): print(f'  {k:16s} {v:.4f}')
print('\nMatriz de confusão:\n', confusion_matrix(test['alvo'], pred))

## 7. Robustez e calibração

In [ ]:
# StratifiedGroupKFold por aluno (evita o mesmo aluno em treino e teste)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
aucs = []
for tr_i, te_i in sgkf.split(tr[FEATURES], tr['alvo'], groups=tr['RA']):
    p = build(); p.fit(tr.iloc[tr_i][FEATURES], tr.iloc[tr_i]['alvo'])
    pp = p.predict_proba(tr.iloc[te_i][FEATURES])[:, 1]
    aucs.append(average_precision_score(tr.iloc[te_i]['alvo'], pp))
auc_cv = float(np.mean(aucs))
print(f'GroupKFold por aluno: AUC-PR = {auc_cv:.4f} ± {np.std(aucs):.4f}')

# Calibração: faixa prevista x taxa real
print('\nCalibração (teste 2023->2024):')
for nome, lo, hi in [('Baixo (0-40%)', 0, .4), ('Moderado (40-70%)', .4, .7), ('Alto (70-100%)', .7, 1.01)]:
    m = (proba >= lo) & (proba < hi)
    if m.sum():
        print(f'  {nome:20s} n={m.sum():4d}  defasagem real em T+1: {test["alvo"][m].mean()*100:5.1f}%')

## 8. Importância das variáveis

In [ ]:
perm = permutation_importance(pipe_final, test[FEATURES], test['alvo'],
                              n_repeats=10, random_state=42, scoring='roc_auc', n_jobs=-1)
imp = (pd.DataFrame({'Variável': FEATURES, 'Δ AUC': perm.importances_mean})
       .sort_values('Δ AUC', ascending=False).reset_index(drop=True))
imp.round(4)

## 9. Exportação do modelo

In [ ]:
final = build()
final.fit(tr[FEATURES], tr['alvo'])   # treina em TODAS as transições

import datetime

bundle = {
    'model': final, 'features': FEATURES, 'num': NUM, 'cat': CAT,
    'threshold': float(thr), 'metricas_teste': metricas, 'auc_pr_groupkfold': auc_cv,
    'target': 'Defasagem_{T+1} < 0 (defasado no ano seguinte)',
    'treino': 'transicoes 2022->2023 + 2023->2024 (RandomForest calibrado)',
    'pedras': sorted([p for p in tr['Pedra'].dropna().unique()]),
    # metadados de transparência — o app exibe na lateral e nos PDFs
    'data_treino': datetime.date.today().isoformat(),
    'versao': 'v0',
}
Path('../models').mkdir(exist_ok=True)
joblib.dump(bundle, '../models/model.joblib')
print('Modelo salvo em ../models/model.joblib')

## 10. Sumário

| Item | Resultado |
|---|---|
| Alvo | Defasagem < 0 no ano **seguinte** (T+1) |
| Dados | 1.365 transições (2022→2023 e 2023→2024), 897 alunos |
| Modelo | RandomForest calibrado |
| Teste out-of-time | AUC-PR 0,822 · AUC-ROC 0,873 · Recall 0,873 |
| Robustez (por aluno) | AUC-PR 0,860 ± 0,029 |
| Calibração | 6,8% → 53,9% → 92,1% (monotônica) |

### Achados principais
1. **Leakage identificado e corrigido**: `Defasagem = Fase − Fase Ideal(Idade)` em 100% dos
   casos. Prever o mesmo ano é aritmética, não previsão.
2. **Ganho real sobre a persistência**: +0,19 de AUC sobre supor que a situação se mantém.
3. **Os índices pedagógicos agregam** +0,042 de AUC além do estado atual — modesto, porém real.
4. **Genero foi removido** por evidência (piorava) e por equidade.
